In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Plot settings
sns.set(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)

data_dir = os.path.join("/Users", "victor_cheng", "Desktop", "UChicago", "Leadership", "Ecommerce EDA", "Ecommerce data")

summary_1819_path = os.path.join(data_dir, "summary2018-19.csv")
summary_2021_path = os.path.join(data_dir, "summary2020-21.csv")
survey_path = os.path.join(data_dir, "survey.csv")

summary_1819 = pd.read_csv(summary_1819_path)
summary_2021 = pd.read_csv(summary_2021_path)
survey = pd.read_csv(survey_path)

print("summary_1819:", summary_1819.shape)
print("summary_2021:", summary_2021.shape)
print("survey:", survey.shape)

summary_1819.head()


In [ ]:
# Basic info about 2018-2019 summary (training period)
print("=== summary_1819 info ===")
summary_1819.info()
print("\n=== summary_1819 describe ===")
print(summary_1819.describe(include="all"))

# Basic info about survey
print("\n=== survey columns ===")
print(survey.columns[:15])
print("Total survey columns:", survey.shape[1])

survey.head()

In [ ]:
# --- Feature engineering: RFM and related metrics for 2018–2019 ---

summary = summary_1819.copy()

# Ensure dates are parsed correctly
summary["firstpurchase"] = pd.to_datetime(summary["firstpurchase"])
summary["lastpurchase"] = pd.to_datetime(summary["lastpurchase"])

# Reference date = end of training window
ref_date = datetime(2019, 12, 31)

summary["recency_days"] = (ref_date - summary["lastpurchase"]).dt.days
summary["tenure_days"] = (summary["lastpurchase"] - summary["firstpurchase"]).dt.days
summary["frequency"] = summary["totalpurchases"].astype(float)
summary["monetary"] = summary["totalspend"].astype(float)
summary["avg_order_value"] = summary["monetary"] / summary["frequency"]

# Log transforms (handle zeros to avoid -inf)
for col in ["frequency", "monetary", "avg_order_value"]:
    summary[f"log_{col}"] = np.log1p(summary[col])

summary[["respondentid", "recency_days", "frequency", "monetary", "avg_order_value"]].head()

In [ ]:
# --- Univariate distributions: Recency, Frequency, Monetary, AOV ---

rfm_cols = ["recency_days", "frequency", "monetary", "avg_order_value"]
summary[rfm_cols].describe()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, col in zip(axes, rfm_cols):
    sns.histplot(summary[col], bins=40, ax=ax, kde=True)
    ax.set_title(col)

plt.tight_layout()
plt.show()

# Boxplots to highlight outliers
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

for ax, col in zip(axes, rfm_cols):
    sns.boxplot(x=summary[col], ax=ax)
    ax.set_title(f"Boxplot: {col}")

plt.tight_layout()
plt.show()


In [ ]:
# --- Bivariate relationships among RFM metrics ---

pairplot_cols = ["recency_days", "log_frequency", "log_monetary", "log_avg_order_value"]

sns.pairplot(summary[pairplot_cols], diag_kind="kde")
plt.suptitle("RFM relationships (log scale for F, M, AOV)", y=1.02)
plt.show()

# Correlation matrix
corr = summary[pairplot_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", vmin=-1, vmax=1)
plt.title("Correlation matrix of RFM features")
plt.show()


In [ ]:
# --- Customer activity over time (approximate) ---

# Distribution of first and last purchase years
summary["first_year"] = summary["firstpurchase"].dt.year
summary["last_year"] = summary["lastpurchase"].dt.year

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(x="first_year", data=summary, ax=axes[0])
axes[0].set_title("Count of customers by first purchase year")

sns.countplot(x="last_year", data=summary, ax=axes[1])
axes[1].set_title("Count of customers by last purchase year")

plt.tight_layout()
plt.show()

# Tenure distribution
sns.histplot(summary["tenure_days"], bins=40, kde=True)
plt.title("Customer tenure (days between first and last purchase)")
plt.show()


In [ ]:
# --- Merge survey with purchase summary ---

survey_renamed = survey.rename(columns={survey.columns[0]: "respondentid"})
merged = pd.merge(summary, survey_renamed, on="respondentid", how="inner")

print("Merged shape:", merged.shape)
merged.head()


In [ ]:
# --- Behavior by key survey segments (examples) ---

# Choose a few interpretable survey fields
age_col = "Q-demos-age"
income_col = "Q-demos-income"
usage_freq_col = "Q-amazon-use-how-oft"

agg_cols = ["recency_days", "frequency", "monetary", "avg_order_value"]

def summarize_by(col_name, label):
    print(f"\n=== RFM by {label} ({col_name}) ===")
    (merged
     .groupby(col_name)[agg_cols]
     .mean()
     .sort_values("monetary", ascending=False)
     .round(2)
     .head(10)
     .pipe(print))

summarize_by(age_col, "age band")
summarize_by(income_col, "income band")
summarize_by(usage_freq_col, "Amazon use frequency")


In [ ]:
# --- Clustering-ready feature set: suggestion for team ---

cluster_features = summary[[
    "recency_days",
    "log_frequency",
    "log_monetary",
    "log_avg_order_value",
    "tenure_days"
]].copy()

print("Cluster feature summary:")
cluster_features.describe().round(2)


# K-Means Clustering Analysis

This section performs K-Means clustering on behavioral features to identify distinct customer segments. This is exploratory work to inform our group's final segmentation decision.


In [ ]:
# --- Prepare data for clustering: Create comprehensive behavioral features ---

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# Start with summary data that has RFM features
# If 'merged' exists (from Cell 7), use it to include survey data; otherwise use summary
if 'merged' in globals():
    df_cluster = merged.copy()
    print("Using merged dataframe (includes survey data)")
else:
    df_cluster = summary.copy()
    print("Using summary dataframe (survey data not yet merged)")

# Additional behavioral metrics (similar to modeling_final approach)
df_cluster["purchase_rate"] = df_cluster["frequency"] / (df_cluster["tenure_days"] + 1)  # purchases per day active
df_cluster["active_rate"] = df_cluster["purchasedays"] / (df_cluster["tenure_days"] + 1)  # active days per day in tenure
df_cluster["spend_per_purchase"] = df_cluster["monetary"] / df_cluster["frequency"]
df_cluster["spend_per_active_day"] = df_cluster["monetary"] / (df_cluster["purchasedays"] + 1)

# Select behavioral features for clustering (similar to modeling_final)
behavior_cols = [
    "totalspend",
    "totalpurchases", 
    "purchasedays",
    "recency_days",
    "purchase_rate",
    "active_rate",
    "spend_per_purchase",
    "spend_per_active_day"
]

# Standardize features (z-score normalization)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_cluster[behavior_cols])
df_cluster_scaled = pd.DataFrame(
    X_scaled, 
    columns=[col + "_scaled" for col in behavior_cols], 
    index=df_cluster.index
)
df_cluster = pd.concat([df_cluster, df_cluster_scaled], axis=1)

print("Behavioral features created and scaled:")
print(f"Shape: {df_cluster.shape}")
print(f"\nScaled features summary:")
print(df_cluster[[col for col in df_cluster.columns if col.endswith("_scaled")]].describe().round(2))


In [ ]:
# --- Elbow Method and Silhouette Analysis for K-Means ---

print("="*80)
print("K-MEANS CLUSTERING: Elbow and Silhouette Analysis")
print("="*80)

# Prepare data matrix
X = df_cluster[[col + "_scaled" for col in behavior_cols]].values

# Try different numbers of clusters
k_list = list(range(2, 9))
inertias = []
sil_scores = []

for k in k_list:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X, labels))
    print(f"k={k}: Inertia={km.inertia_:.2f}, Silhouette={sil_scores[-1]:.4f}")

print(f"\nInertia by K: {dict(zip(k_list, inertias))}")
print(f"Silhouette by K: {dict(zip(k_list, sil_scores))}")

# Plot elbow and silhouette
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Elbow plot
ax[0].plot(k_list, inertias, marker="o")
ax[0].set_xlabel("K")
ax[0].set_ylabel("Inertia (SSE)")
ax[0].set_title("Elbow Plot")
ax[0].grid(True, alpha=0.3)

# Silhouette plot
ax[1].plot(k_list, sil_scores, marker="o")
ax[1].set_xlabel("K")
ax[1].set_ylabel("Silhouette score")
ax[1].set_title("Silhouette vs K")
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Choose best K (highest silhouette score)
best_k = k_list[np.argmax(sil_scores)]
best_sil = max(sil_scores)
print(f"\nBest K based on silhouette: {best_k} (Silhouette: {best_sil:.4f})")


## Comparison: k=3 vs k=4 Clustering

This section compares both k=3 and k=4 to help decide which provides better business segmentation.


In [ ]:
# --- Compare k=3 vs k=4: Cluster Profiles and Business Interpretation ---

print("="*80)
print("COMPARISON: k=3 vs k=4 CLUSTERING")
print("="*80)

# Prepare data matrix (same as before)
X = df_cluster[[col + "_scaled" for col in behavior_cols]].values

# Fit both k=3 and k=4
kmeans_k3 = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans_k4 = KMeans(n_clusters=4, random_state=42, n_init=10)

df_compare = df_cluster.copy()
df_compare["cluster_k3"] = kmeans_k3.fit_predict(X)
df_compare["cluster_k4"] = kmeans_k4.fit_predict(X)

# Calculate silhouette scores
sil_k3 = silhouette_score(X, df_compare["cluster_k3"])
sil_k4 = silhouette_score(X, df_compare["cluster_k4"])

print(f"\nSilhouette Scores:")
print(f"  k=3: {sil_k3:.4f}")
print(f"  k=4: {sil_k4:.4f}")
print(f"  Difference: {abs(sil_k3 - sil_k4):.4f}")

# Cluster sizes
print(f"\nCluster Sizes:")
print(f"\nk=3:")
print(df_compare["cluster_k3"].value_counts().sort_index())
print(f"\nk=4:")
print(df_compare["cluster_k4"].value_counts().sort_index())

# Key business metrics for comparison
key_metrics = ["totalspend", "totalpurchases", "purchasedays", "recency_days", 
               "purchase_rate", "spend_per_purchase"]

print("\n" + "="*80)
print("CLUSTER PROFILES: k=3")
print("="*80)
profile_k3 = df_compare.groupby("cluster_k3")[key_metrics].agg(['mean', 'median', 'count']).round(2)
print(profile_k3)

print("\n" + "="*80)
print("CLUSTER PROFILES: k=4")
print("="*80)
profile_k4 = df_compare.groupby("cluster_k4")[key_metrics].agg(['mean', 'median', 'count']).round(2)
print(profile_k4)

In [ ]:
# --- Visual Comparison: k=3 vs k=4 ---

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Total Spend vs Total Purchases - k=3
ax = axes[0, 0]
for cluster in sorted(df_compare["cluster_k3"].unique()):
    cluster_data = df_compare[df_compare["cluster_k3"] == cluster]
    ax.scatter(cluster_data["totalpurchases"], cluster_data["totalspend"], 
              label=f'Cluster {cluster}', alpha=0.6, s=50)
ax.set_xlabel('Total Purchases')
ax.set_ylabel('Total Spend ($)')
ax.set_title(f'k=3 Clusters (Silhouette: {sil_k3:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# 2. Total Spend vs Total Purchases - k=4
ax = axes[0, 1]
for cluster in sorted(df_compare["cluster_k4"].unique()):
    cluster_data = df_compare[df_compare["cluster_k4"] == cluster]
    ax.scatter(cluster_data["totalpurchases"], cluster_data["totalspend"], 
              label=f'Cluster {cluster}', alpha=0.6, s=50)
ax.set_xlabel('Total Purchases')
ax.set_ylabel('Total Spend ($)')
ax.set_title(f'k=4 Clusters (Silhouette: {sil_k4:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# 3. Boxplot: Total Spend by cluster - k=3
ax = axes[1, 0]
df_compare.boxplot(column='totalspend', by='cluster_k3', ax=ax)
ax.set_xlabel('Cluster (k=3)')
ax.set_ylabel('Total Spend ($)')
ax.set_title('Total Spend Distribution by Cluster (k=3)')
ax.set_xticklabels([f'Cluster {i}' for i in sorted(df_compare["cluster_k3"].unique())])

# 4. Boxplot: Total Spend by cluster - k=4
ax = axes[1, 1]
df_compare.boxplot(column='totalspend', by='cluster_k4', ax=ax)
ax.set_xlabel('Cluster (k=4)')
ax.set_ylabel('Total Spend ($)')
ax.set_title('Total Spend Distribution by Cluster (k=4)')
ax.set_xticklabels([f'Cluster {i}' for i in sorted(df_compare["cluster_k4"].unique())])

plt.suptitle('Comparison: k=3 vs k=4 Clustering', fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

In [ ]:
# --- Business Interpretation: Four-Quadrant Framework (k=4) ---

print("="*80)
print("BUSINESS INTERPRETATION: k=4 Four-Quadrant Framework")
print("="*80)

# Calculate median frequency and spending for k=4 to create quadrants
median_freq = df_compare.groupby("cluster_k4")["totalpurchases"].mean().median()
median_spend = df_compare.groupby("cluster_k4")["totalspend"].mean().median()

print(f"\nMedian Frequency (purchases): {median_freq:.1f}")
print(f"Median Spending: ${median_spend:.2f}\n")

# Classify each cluster into quadrants
cluster_summary_k4 = df_compare.groupby("cluster_k4").agg({
    "totalspend": "mean",
    "totalpurchases": "mean",
    "purchasedays": "mean",
    "recency_days": "mean"
}).round(2)

cluster_summary_k4["cluster_size"] = df_compare.groupby("cluster_k4").size()
cluster_summary_k4["pct_of_total"] = (cluster_summary_k4["cluster_size"] / len(df_compare) * 100).round(1)

print("Cluster Classification (Four-Quadrant Framework):")
print("-" * 80)

for cluster_id in sorted(cluster_summary_k4.index):
    freq = cluster_summary_k4.loc[cluster_id, "totalpurchases"]
    spend = cluster_summary_k4.loc[cluster_id, "totalspend"]
    recency = cluster_summary_k4.loc[cluster_id, "recency_days"]
    size = cluster_summary_k4.loc[cluster_id, "cluster_size"]
    pct = cluster_summary_k4.loc[cluster_id, "pct_of_total"]
    
    # Classify into quadrant
    if freq >= median_freq and spend >= median_spend:
        quadrant = "High Frequency, High Spending (Champions)"
        strategy = "Category-matched recommendations, no discounts, loyalty programs"
    elif freq < median_freq and spend >= median_spend:
        quadrant = "Low Frequency, High Spending (Whales)"
        strategy = "Win-back campaigns, personalized high-value offers"
    elif freq >= median_freq and spend < median_spend:
        quadrant = "High Frequency, Low Spending (Bargain Hunters)"
        strategy = "Volume-based promotions, bundling, increase basket size"
    else:
        quadrant = "Low Frequency, Low Spending (At-Risk/Inactive)"
        strategy = "Re-engagement campaigns, strong discounts, popular items"
    
    print(f"\nCluster {cluster_id} ({size} customers, {pct}%):")
    print(f"  Quadrant: {quadrant}")
    print(f"  Avg Total Spend: ${spend:.2f}")
    print(f"  Avg Total Purchases: {freq:.1f}")
    print(f"  Avg Recency (days): {recency:.1f}")
    print(f"  Strategy: {strategy}")

print("\n" + "="*80)
print("RECOMMENDATION")
print("="*80)
print(f"\nBased on silhouette scores:")
print(f"  - k=3: {sil_k3:.4f}")
print(f"  - k=4: {sil_k4:.4f}")
print(f"  - Difference: {abs(sil_k3 - sil_k4):.4f}")

if abs(sil_k3 - sil_k4) < 0.05:
    print("\n✓ The silhouette scores are very close (< 0.05 difference).")
    print("  Consider k=4 for better business interpretability (four-quadrant framework).")
elif sil_k3 > sil_k4:
    print(f"\n✓ k=3 has a higher silhouette score (+{abs(sil_k3 - sil_k4):.4f}).")
    print("  However, k=4 provides the four-quadrant framework which is:")
    print("  - More interpretable for business stakeholders")
    print("  - Aligned with classic RFM segmentation")
    print("  - Easier to explain in presentations")
    print("\n  Recommendation: Use k=4 unless k=3 shows clearly distinct business segments.")
else:
    print(f"\n✓ k=4 has a higher silhouette score (+{abs(sil_k4 - sil_k3):.4f}).")
    print("  Recommendation: Use k=4 (both statistically and business-wise optimal).")

In [ ]:
# --- Fit K-Means with best K and assign clusters ---
best_k = 4

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = kmeans.fit_predict(X)

print(f"Final K used = {best_k}")
print(f"\nCluster distribution:")
print(df_cluster["cluster"].value_counts().sort_index())

# Cluster summary statistics
print(f"\nCluster Summary Statistics:")
cluster_summary = df_cluster.groupby("cluster")[behavior_cols].agg(['mean', 'median', 'std']).round(2)
print(cluster_summary)

In [ ]:
# --- PCA 2D Visualization ---

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(7, 6))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=df_cluster["cluster"],
    palette="tab10",
    alpha=0.6
)
plt.title(f"PCA Projection with KMeans Clusters (K={best_k})")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.2%}")


In [ ]:
# --- Cluster Behavioral Profiles (Radar Chart) ---

# Calculate cluster means for scaled features
behavior_cols_scaled = [col + "_scaled" for col in behavior_cols]
cluster_profile = df_cluster.groupby("cluster")[behavior_cols_scaled].mean()

# Radar chart
labels = behavior_cols_scaled
num_vars = len(labels)
angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
angles += angles[:1]  # close the loop

plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

for c in cluster_profile.index:
    values = cluster_profile.loc[c].tolist()
    values += values[:1]  # close the loop
    ax.plot(angles, values, label=f"Cluster {c}", linewidth=2)
    ax.fill(angles, values, alpha=0.1)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=8)
ax.set_title("Cluster Behavioral Profiles (Radar Chart)", y=1.08)
ax.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.show()

print("\nCluster behavioral means (z-scores):")
print(cluster_profile.round(2))


In [ ]:
# --- Bar plots for key behavioral variables by cluster ---

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.ravel()

for idx, col in enumerate(behavior_cols_scaled):
    sns.barplot(x="cluster", y=col, data=df_cluster, ax=axes[idx])
    axes[idx].set_title(f"{col} by Cluster")
    axes[idx].set_xlabel("Cluster")
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# --- Boxplots for key business variables by cluster ---

key_vars = ["totalspend_scaled", "purchase_rate_scaled", "spend_per_purchase_scaled", "recency_days_scaled"]

for col in key_vars:
    plt.figure(figsize=(8, 6))
    sns.boxplot(data=df_cluster, x="cluster", y=col, palette="Set2")
    sns.stripplot(data=df_cluster, x="cluster", y=col, color="black", alpha=0.15, size=3)
    plt.title(f"Distribution of {col} by K-Means Cluster")
    plt.xlabel("Cluster")
    plt.ylabel(f"{col} (standardized)")
    plt.tight_layout()
    plt.show()

In [ ]:
# --- Merge survey data and analyze survey variables by cluster ---

# Merge survey with clustered data
survey_renamed = survey.rename(columns={survey.columns[0]: "respondentid"})
df_cluster_merged = pd.merge(df_cluster, survey_renamed, on="respondentid", how="inner")

print(f"Merged shape: {df_cluster_merged.shape}")

# Select key survey columns for analysis (categorical variables)
survey_cols = [c for c in df_cluster_merged.columns if 
               c.startswith("Q-demos-income") or 
               c.startswith("Q-amazon-use-how-oft") or 
               c.startswith("Q-demos-age") or 
               c.startswith("Q-demos-education")]

if survey_cols:
    # For categorical variables, we need to use one-hot encoding or crosstab
    # Let's create a heatmap showing the proportion of each category within each cluster
    
    # Create a list to store crosstab results
    crosstab_results = []
    
    for col in survey_cols:
        # Create crosstab: cluster vs survey category (normalized by cluster)
        crosstab = pd.crosstab(df_cluster_merged["cluster"], df_cluster_merged[col], normalize='index') * 100
        crosstab_results.append((col, crosstab))
    
    # Create a combined visualization for key survey variables
    if len(crosstab_results) > 0:
        # Plot first few survey variables
        n_vars = min(4, len(crosstab_results))
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        axes = axes.ravel()
        
        for idx in range(n_vars):
            col_name, crosstab = crosstab_results[idx]
            sns.heatmap(crosstab, cmap="Blues", annot=True, fmt='.1f', ax=axes[idx], cbar_kws={'label': '%'})
            axes[idx].set_title(f"{col_name}\n(Proportion by Cluster)", fontsize=10)
            axes[idx].set_xlabel("Category", fontsize=9)
            axes[idx].set_ylabel("Cluster", fontsize=9)
            axes[idx].tick_params(axis='x', rotation=45, labelsize=7)
            axes[idx].tick_params(axis='y', labelsize=8)
        
        plt.tight_layout()
        plt.show()
        
        # Print summary statistics
        print("\nSurvey variable distribution by cluster (top categories):")
        for col_name, crosstab in crosstab_results[:4]:  # Show first 4
            print(f"\n{col_name}:")
            # Show top category for each cluster
            for cluster_id in crosstab.index:
                top_cat = crosstab.loc[cluster_id].idxmax()
                top_pct = crosstab.loc[cluster_id, top_cat]
                print(f"  Cluster {cluster_id}: {top_cat} ({top_pct:.1f}%)")
else:
    print("No matching survey columns found")


In [ ]:
# --- 3D PCA Visualization ---

from mpl_toolkits.mplot3d import Axes3D

pca_3d = PCA(n_components=3)
X_pca_3d = pca_3d.fit_transform(X)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

for c in sorted(df_cluster["cluster"].unique()):
    mask = df_cluster["cluster"] == c
    ax.scatter(
        X_pca_3d[mask, 0],
        X_pca_3d[mask, 1],
        X_pca_3d[mask, 2],
        s=20,
        alpha=0.6,
        label=f"Cluster {c}",
    )

ax.set_title("K-Means Clusters in 3D PCA Space")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
ax.legend()
plt.tight_layout()
plt.show()

print(f"3D PCA explained variance ratio: {pca_3d.explained_variance_ratio_}")
print(f"Total variance explained: {pca_3d.explained_variance_ratio_.sum():.2%}")